## Exercise 2

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

spark = (
    SparkSession.builder
    .appName("Session09Part02")
    .master("local[*]")
    .getOrCreate()
)

schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("request_count", IntegerType(), True),
    StructField("error_count", IntegerType(), True),
    StructField("latency_ms", DoubleType(), True),
    StructField("bytes_in", DoubleType(), True),
    StructField("bytes_out", DoubleType(), True),
])

events_df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv("../datasets/service_events.csv")
)

In [3]:
from pyspark.sql.functions import col, round, when

events_enriched_df = (
    events_df
    .withColumn(
        "error_rate",
        when(col("request_count") > 0, col("error_count") / col("request_count")).otherwise(0),
    )
    .withColumn("total_bytes", col("bytes_in") + col("bytes_out"))
    .withColumn("traffic_mb", col("total_bytes") / 1048576)
)

events_enriched_df.select(
    "service",
    "request_count",
    "error_count",
    round("error_rate", 4).alias("error_rate"),
    round("traffic_mb", 2).alias("traffic_mb"),
).show(8)

+---------------+-------------+-----------+----------+----------+
|        service|request_count|error_count|error_rate|traffic_mb|
+---------------+-------------+-----------+----------+----------+
|           auth|         1240|          8|    0.0065|     50.26|
|       payments|          860|         21|    0.0244|     60.37|
|         search|         2110|         13|    0.0062|    100.14|
|recommendations|         1680|         18|    0.0107|    120.07|
|           auth|          980|          4|    0.0041|     39.77|
|       payments|          740|         17|     0.023|     58.27|
|         search|         2320|         15|    0.0065|    109.77|
|recommendations|         1760|         24|    0.0136|    128.75|
+---------------+-------------+-----------+----------+----------+
only showing top 8 rows


In [4]:
events_enriched_df = events_enriched_df.withColumn(
    "latency_band",
    when(col("latency_ms") < 100, "fast")
    .when(col("latency_ms") < 160, "normal")
    .otherwise("slow"),
)

events_enriched_df.select("service", "latency_ms", "latency_band").show(8)

+---------------+----------+------------+
|        service|latency_ms|latency_band|
+---------------+----------+------------+
|           auth|      82.4|        fast|
|       payments|     146.2|      normal|
|         search|      94.8|        fast|
|recommendations|     132.5|      normal|
|           auth|      78.1|        fast|
|       payments|     171.3|        slow|
|         search|     101.7|      normal|
|recommendations|     155.9|      normal|
+---------------+----------+------------+
only showing top 8 rows


In [5]:
from pyspark.sql.functions import date_format, hour, to_date

events_enriched_df = (
    events_enriched_df
    .withColumn("event_date", to_date(col("event_time")))
    .withColumn("event_hour", hour(col("event_time")))
    .withColumn("day_of_week", date_format(col("event_time"), "E"))
)

events_enriched_df.select(
    "event_time",
    "event_date",
    "event_hour",
    "day_of_week",
).show(8, truncate=False)

+-------------------+----------+----------+-----------+
|event_time         |event_date|event_hour|day_of_week|
+-------------------+----------+----------+-----------+
|2026-05-04 00:00:00|2026-05-04|0         |Mon        |
|2026-05-04 00:00:00|2026-05-04|0         |Mon        |
|2026-05-04 01:00:00|2026-05-04|1         |Mon        |
|2026-05-04 01:00:00|2026-05-04|1         |Mon        |
|2026-05-04 02:00:00|2026-05-04|2         |Mon        |
|2026-05-04 02:00:00|2026-05-04|2         |Mon        |
|2026-05-04 03:00:00|2026-05-04|3         |Mon        |
|2026-05-04 03:00:00|2026-05-04|3         |Mon        |
+-------------------+----------+----------+-----------+
only showing top 8 rows


In [6]:
events_enriched_df.createOrReplaceTempView("service_events_enriched")

In [10]:
spark.sql("""
    SELECT
        service,
        ROUND(AVG(error_rate), 4) AS average_error_rate,
        ROUND(AVG(latency_ms), 2) AS average_latency_ms,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY service
    ORDER BY average_error_rate DESC
""").show()

+---------------+------------------+------------------+----------------+
|        service|average_error_rate|average_latency_ms|total_traffic_mb|
+---------------+------------------+------------------+----------------+
|       payments|            0.0243|            174.63|          407.89|
|recommendations|            0.0119|            140.75|          723.65|
|         search|            0.0057|             94.02|          614.26|
|           auth|            0.0051|             79.63|          305.56|
+---------------+------------------+------------------+----------------+



In [21]:
spark.sql("""
    SELECT
        event_hour,
        SUM(request_count) AS total_requests
    FROM service_events_enriched
    GROUP BY event_hour
    ORDER BY total_requests DESC
""").show()

+----------+--------------+
|event_hour|total_requests|
+----------+--------------+
|         3|          8310|
|         1|          7700|
|        10|          6950|
|         9|          5340|
|         0|          4320|
|         2|          3510|
+----------+--------------+



In [19]:
spark.sql("""
    SELECT
        region,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY region
    ORDER BY total_traffic_mb DESC
""").show()

+--------+----------------+
|  region|total_traffic_mb|
+--------+----------------+
| us-east|          733.76|
| eu-west|          710.87|
|ap-south|          606.73|
+--------+----------------+



In [20]:
events_enriched_df.show(10)

+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+--------------------+-----------+------------------+------------+----------+----------+-----------+
|event_id|        service|  region|         event_time|request_count|error_count|latency_ms|bytes_in|bytes_out|          error_rate|total_bytes|        traffic_mb|latency_band|event_date|event_hour|day_of_week|
+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+--------------------+-----------+------------------+------------+----------+----------+-----------+
|       1|           auth| eu-west|2026-05-04 00:00:00|         1240|          8|      82.4|  1.45E7|   3.82E7|0.006451612903225...|     5.27E7|50.258636474609375|        fast|2026-05-04|         0|        Mon|
|       2|       payments| eu-west|2026-05-04 00:00:00|          860|         21|     146.2|  1.92E7|   4.41E7| 0.02441860465116279|     6.33E7|60.367584228

In [22]:
spark.stop()